# 10 — Mini GPT: A Transformer Language Model from Scratch

In Project 09, you built an **LSTM** that writes Shakespeare one character at a time. It worked — but LSTM's process sequentially: character 1, then 2, then 3... It has no way to look directly from character 50 back to character 3 without going through every character in between.

**Transformers** changed everything. Instead of processing one character at a time, they look at **all positions simultaneously** using a mechanism called **self-attention**. Every character can directly "attend to" every other character in the sequence.

This is the architecture behind GPT, BERT, Claude, and every modern LLM.

**What you'll learn:**
1. **Self-Attention** — the core innovation: every token looks at every other token
2. **Multi-Head Attention** — multiple attention patterns running in parallel
3. **Positional Encoding** — how the model knows word order without recurrence
4. **Causal Masking** — preventing the model from "cheating" by looking ahead
5. **Layer Norm & Residual Connections** — the engineering tricks that make deep transformers trainable
6. **Building a decoder-only transformer** — the same architecture as GPT-2

**The dataset:** Same Shakespeare we used in Project 09 — so you can directly compare LSTM vs Transformer!

## Setup

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
import urllib.request
import time
import math

# Device detection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Random seed set to {SEED}')

---
## Part 1: Load the Shakespeare Dataset

Same exact dataset as Project 09 — this gives us a direct comparison. We'll use the same preprocessing pipeline.

In [ ]:
data_dir = Path.home() / "LocalAI" / "data" / "shakespeare"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "shakespeare.txt"
url = "https://www.gutenberg.org/cache/epub/100/pg100.txt"

if not file_path.exists():
    print("Downloading Shakespeare's complete works (~5.5MB)...")
    urllib.request.urlretrieve(url, file_path)
    print("Done!")
else:
    print("Shakespeare already downloaded (shared with Project 09).")

# Read and clean (same as Project 09)
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    start_idx = raw_text.find('\n', start_idx) + 1
    text = raw_text[start_idx:end_idx].strip()
else:
    text = raw_text

print(f'Cleaned text: {len(text):,} characters')
print(f'Unique characters: {len(set(text)):,}')
print(f'\nFirst 300 chars:\n{text[:300]}')

---
## Part 2: Character-Level Tokenization

Same approach as Project 09. We keep this identical so the comparison is fair — the only difference is LSTM vs Transformer.

In [ ]:
# Build character vocabulary (same as Project 09)
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f'Vocabulary size: {vocab_size} unique characters')

# Create mappings
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Convert entire text to integer IDs
text_as_ids = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)
print(f'Text as integers: {len(text_as_ids):,} tokens')
print(f'First 80 IDs: {text_as_ids[:80]}')

---
## Part 3: The Core Idea — Next-Character Prediction

**Same setup as Project 09.** The model's job: given N characters, predict what comes next.

The fundamental difference is **how** the model processes those N characters:

| | LSTM (Project 09) | Transformer (This Project) |
|--|----|----|
| Processing | Sequential: char 1 → char 2 → char 3 → ... | Parallel: all chars at once |
| How it sees context | Through a hidden state that gets updated one step at a time | Via self-attention: every char directly connects to every other char |
| Long-range dependencies | Struggles beyond ~50 chars (gradient issues) | Excellent at any distance within context window |
| Training speed | Slow (can't parallelize over sequence) | Fast (all positions processed simultaneously) |

In [ ]:
SEQUENCE_LENGTH = 128  # How many characters the model sees to predict the next one
BATCH_SIZE = 64        # Smaller batches than LSTM — transformers use more memory

class CharDataset(Dataset):
    """Same sliding-window dataset as Project 09."""
    def __init__(self, text_ids, seq_length):
        self.text_ids = text_ids
        self.seq_length = seq_length
    
    def __len__(self):
        return (len(self.text_ids) - self.seq_length) // self.seq_length
    
    def __getitem__(self, idx):
        start = idx * self.seq_length
        input_seq = self.text_ids[start : start + self.seq_length]
        target_seq = self.text_ids[start + 1 : start + self.seq_length + 1]
        return (
            torch.tensor(input_seq, dtype=torch.long),
            torch.tensor(target_seq, dtype=torch.long)
        )

# Train/val split
split_idx = int(len(text_as_ids) * 0.9)
train_ids = text_as_ids[:split_idx]
val_ids = text_as_ids[split_idx:]

train_dataset = CharDataset(train_ids, SEQUENCE_LENGTH)
val_dataset = CharDataset(val_ids, SEQUENCE_LENGTH)

print(f'Training examples:   {len(train_dataset):,}')
print(f'Validation examples: {len(val_dataset):,}')

pin = device.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=4, pin_memory=pin)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=pin)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

---
## Part 4: Understanding Self-Attention — The Key Innovation

This is the most important section in this notebook. Take your time.

### The Problem Self-Attention Solves

Imagine the sentence: **"The cat, which was very hungry, sat on the mat."**

An LSTM reads this word by word. By the time it reaches "sat", it has to remember "cat" from 6 words ago — through a compressed hidden state. Information gets diluted.

Self-attention lets "sat" look **directly** at "cat" (and every other word) and ask: *"How relevant are you to predicting what I should be?"*

### How It Works (Intuition)

Every token creates three vectors from itself:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?"
- **Value (V):** "What information do I pass along if you attend to me?"

To figure out how much token `i` should attend to token `j`:
1. Take token `i`'s **query** and dot-product it with token `j`'s **key**
2. High score = token `j` is very relevant to token `i`
3. Softmax all those scores to get attention weights
4. Compute weighted sum of all **values** using those weights

### The Math

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

The $\sqrt{d_k}$ scaling prevents the dot products from getting too large (which would push softmax into extreme values).

In [ ]:
# Let's visualize attention with a concrete example
print("SELF-ATTENTION INTUITION")
print("=" * 60)
print()
print('Sequence: "The cat sat on the mat"')
print()
print('When processing the word "sat", the model computes:')
print()
print('  Query("sat") · Key("The")  → 0.12  (low relevance)')
print('  Query("sat") · Key("cat")  → 0.45  (high — "cat" is the subject!)')
print('  Query("sat") · Key("sat")  → 0.20  (self-attention)')
print('  Query("sat") · Key("on")   → 0.15  (medium — preposition)')
print('  Query("sat") · Key("the")  → 0.05  (low)')
print('  Query("sat") · Key("mat")  → 0.03  (low)')
print()
print('Weighted sum: ~0.45 * Value("cat") + smaller contributions from others')
print()
print('The output for position "sat" is heavily influenced by "cat" —')
print('the model learned that subjects and verbs should attend to each other!')

---
## Part 5: Building the Transformer — Piece by Piece

We'll build the entire GPT-style transformer from scratch. Each component is surprisingly simple on its own.

### 5.1 Single-Head Self-Attention

The purest form of attention. One set of Q, K, V projections.

In [ ]:
class SelfAttention(nn.Module):
    """Single-head self-attention (for learning purposes)."""
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key   = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.head_dim = head_dim
    
    def forward(self, x, causal_mask=True):
        """
        x: (batch, seq_len, embed_dim)
        Returns: (batch, seq_len, head_dim)
        """
        B, T, _ = x.shape
        
        # Step 1: Compute Q, K, V
        Q = self.query(x)  # (B, T, head_dim)
        K = self.key(x)    # (B, T, head_dim)
        V = self.value(x)  # (B, T, head_dim)
        
        # Step 2: Compute attention scores
        # Q @ K^T -> (B, T, T) — a score for every pair of positions
        scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, T, T)
        
        # Step 3: Scale to prevent extreme softmax values
        scores = scores / math.sqrt(self.head_dim)
        
        # Step 4: Causal mask — prevent looking at future tokens
        if causal_mask:
            # Create an upper-triangular mask (True = masked out)
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(mask, float('-inf'))
        
        # Step 5: Softmax -> attention weights
        attn_weights = F.softmax(scores, dim=-1)  # (B, T, T)
        
        # Step 6: Weighted sum of values
        out = torch.matmul(attn_weights, V)  # (B, T, head_dim)
        
        return out

print('SelfAttention module defined!')
print()
print('Key insight: The causal mask is an upper-triangular matrix of -inf.')
print('This means position 5 can only attend to positions 0-5, not 6+.')
print('This is what makes it a *generative* model — it can only use the past.')

### 5.2 Multi-Head Attention

Instead of one attention pattern, run **multiple in parallel** then concatenate. Each "head" can learn different types of relationships:
- Head 1: attends to nearby words (local syntax)
- Head 2: attends to the subject of the sentence (long-range dependency)
- Head 3: attends to punctuation and sentence boundaries
- etc.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention — the core of the Transformer."""
    def __init__(self, embed_dim, n_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % n_heads == 0, f'embed_dim ({embed_dim}) must be divisible by n_heads ({n_heads})'
        
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.embed_dim = embed_dim
        
        # Combined Q,K,V projection — more efficient than separate ones
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        
        # Project concatenated heads back to embed_dim
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, causal_mask=True):
        """
        x: (B, T, embed_dim)
        Returns: (B, T, embed_dim)
        """
        B, T, C = x.shape
        
        # Single projection, then split into Q, K, V
        qkv = self.qkv(x)  # (B, T, 3*C)
        qkv = qkv.reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, n_heads, T, head_dim)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # Attention scores: (B, n_heads, T, head_dim) @ (B, n_heads, head_dim, T) -> (B, n_heads, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal mask
        if causal_mask:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(mask, float('-inf'))
        
        # Softmax + dropout on attention weights
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Weighted sum: (B, n_heads, T, T) @ (B, n_heads, T, head_dim) -> (B, n_heads, T, head_dim)
        out = torch.matmul(attn_weights, V)
        
        # Re-assemble all heads: (B, n_heads, T, head_dim) -> (B, T, C)
        out = out.transpose(1, 2).contiguous().reshape(B, T, C)
        
        # Final output projection
        out = self.out_proj(out)
        
        return out

print('MultiHeadAttention module defined!')
print()
print('Each head operates on a smaller slice (head_dim = embed_dim / n_heads).')
print('The heads are concatenated back together and projected to the original dimension.')

### 5.3 Feed-Forward Network

After attention gathers information from all positions, the FFN processes it independently at each position. This is where most of the model's "knowledge" lives.

Standard transformer FFN: two linear layers with an expansion factor (typically 4×).

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    def __init__(self, embed_dim, expansion_factor=4, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * expansion_factor
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),  # GPT uses GELU instead of ReLU
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

print('FeedForward module defined!')
print()
print('The FFN expands 4x (embed_dim → 4*embed_dim → embed_dim).')
print('This is where ~2/3 of the model parameters live.')

### 5.4 Transformer Block

One layer of the transformer: Attention → Add & Norm → FFN → Add & Norm.

The **residual connections** (Add) are critical — they allow gradients to flow directly through the network, making deep transformers trainable. The **layer norm** stabilizes training.

In [ ]:
class TransformerBlock(nn.Module):
    """One transformer decoder block (GPT-style)."""
    def __init__(self, embed_dim, n_heads, expansion_factor=4, dropout=0.1):
        super().__init__()
        
        # Self-attention with residual connection and layer norm
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, dropout)
        
        # Feed-forward with residual connection and layer norm
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, expansion_factor, dropout)
    
    def forward(self, x):
        # Pre-LayerNorm architecture (used in modern GPT variants)
        # Attention sub-layer
        x = x + self.attn(self.ln1(x))
        # Feed-forward sub-layer
        x = x + self.ffn(self.ln2(x))
        return x

print('TransformerBlock module defined!')
print()
print('The pattern is always: x = x + sublayer(layer_norm(x))')
print('The "+" is a residual connection — it adds the input back to the output.')
print('This lets gradients flow directly through, solving the vanishing gradient problem.')

### 5.5 Positional Encoding

Transformers have no recurrence or convolution — they process all positions in parallel. So how do they know the order of tokens?

**Answer:** We add position information directly to the input embeddings.

We use sinusoidal positional encodings (from the original "Attention is All You Need" paper). Each position gets a unique pattern of sine and cosine waves at different frequencies.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding — injects position information."""
    def __init__(self, embed_dim, max_len=1024, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix: (max_len, embed_dim)
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        
        # div_term: different frequencies for different dimensions
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * 
            (-math.log(10000.0) / embed_dim)
        )
        
        # Even dimensions: sin; Odd dimensions: cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension: (1, max_len, embed_dim)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        # x: (B, seq_len, embed_dim)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

print('PositionalEncoding module defined!')

# Visualize positional encodings
pe_module = PositionalEncoding(128, max_len=200)
pe = pe_module.pe[0, :200, :128].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full heatmap
im = axes[0].imshow(pe.T, aspect='auto', cmap='RdBu_r')
axes[0].set_title('Positional Encodings\n(128 dims × 200 positions)', fontsize=13)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Embedding Dimension')
plt.colorbar(im, ax=axes[0])

# First few dimensions as waveforms
for d in [0, 4, 8, 16, 32]:
    axes[1].plot(pe[:200, d], label=f'dim {d}')
axes[1].set_title('Select Dimensions as Waveforms', fontsize=13)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Encoding Value')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Notice: Low dimensions = high frequency (fast oscillation).')
print('High dimensions = low frequency (slow oscillation).')
print('Each position gets a unique "fingerprint" across these dimensions.')

### 5.6 The Full GPT Model

Put it all together:
1. **Token Embedding** — each character → learnable vector
2. **Positional Encoding** — add position information
3. **N Transformer Blocks** — stacked layers of attention + FFN
4. **Final Layer Norm** — stabilize before prediction
5. **LM Head** — project to vocabulary size, predict next character

In [ ]:
class MiniGPT(nn.Module):
    """A GPT-style decoder-only transformer.
    
    This has the same architecture as GPT-2, just much smaller:
    - Decoder-only (no encoder, no cross-attention)
    - Causal self-attention (can only look at past tokens)
    - Pre-layer norm
    - GELU activations
    """
    def __init__(
        self,
        vocab_size,
        embed_dim=256,
        n_heads=8,
        n_layers=6,
        expansion_factor=4,
        max_seq_len=256,
        dropout=0.1
    ):
        super().__init__()
        
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len
        
        # Token embedding: character ID -> learnable vector
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Positional encoding: inject position information
        self.positional_encoding = PositionalEncoding(embed_dim, max_seq_len, dropout)
        
        # Stack of transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, n_heads, expansion_factor, dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm (GPT-2 style)
        self.ln_final = nn.LayerNorm(embed_dim)
        
        # Language model head: project to vocabulary
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        
        # Weight tying: share weights between embedding and output projection
        # This saves parameters and improves performance
        self.token_embedding.weight = self.lm_head.weight
        
        # Initialize weights properly
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x):
        """
        x: (B, T) integer token IDs
        Returns: (B, T, vocab_size) logits
        """
        # x shape: (B, T)
        
        # Token embeddings: (B, T) -> (B, T, embed_dim)
        x = self.token_embedding(x)
        
        # Add positional encoding
        x = self.positional_encoding(x)
        
        # Pass through all transformer blocks
        x = self.blocks(x)
        
        # Final layer norm
        x = self.ln_final(x)
        
        # Project to vocabulary: (B, T, embed_dim) -> (B, T, vocab_size)
        logits = self.lm_head(x)
        
        return logits

# Instantiate the model
model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=256,
    n_heads=8,
    n_layers=6,
    max_seq_len=SEQUENCE_LENGTH,
    dropout=0.1
).to(device)

print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nMiniGPT parameters:')
print(f'  Total:      {total_params:,}')
print(f'  Trainable:  {trainable_params:,}')
print()
print(f'Project 09 LSTM had ~1,000,000 parameters')
print(f'MiniGPT has      ~{total_params/1e6:.1f}M parameters')
print(f'GPT-2 Small has  ~124M parameters')
print(f'GPT-3 has        ~175,000M parameters')
print()
print(f'We are {total_params / 175e9 * 100:.7f}% the size of GPT-3 😄')

---
## Part 6: Training the Transformer

The training loop is nearly identical to Project 09 — same loss function, same metric (perplexity). But transformers train very differently:

- **Faster per epoch** — all positions processed in parallel
- **Needs warmup** — transformers benefit from gradually increasing the learning rate
- **More sensitive to learning rate** — too high and they diverge

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Cosine annealing with warmup — standard for transformers
# We warm up for 20% of training, then cosine-decay to near-zero
N_EPOCHS = 15
warmup_epochs = 3

# Learning rate scheduler: linear warmup + cosine decay
def get_lr(epoch, warmup_epochs, total_epochs, base_lr=3e-4):
    if epoch < warmup_epochs:
        # Linear warmup
        return base_lr * (epoch + 1) / warmup_epochs
    else:
        # Cosine decay
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1 + math.cos(math.pi * progress))

train_losses = []
val_losses = []
val_perplexities = []

print(f'Training on {len(train_loader)} batches per epoch...')
print(f'Model: MiniGPT ({total_params:,} params)')
print(f'LR schedule: linear warmup ({warmup_epochs} epochs) + cosine decay')
print()

start_time = time.time()

for epoch in range(N_EPOCHS):
    # --- TRAINING ---
    model.train()
    epoch_loss = 0
    
    # Set learning rate for this epoch
    current_lr = get_lr(epoch, warmup_epochs, N_EPOCHS)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr
    
    for batch_input, batch_target in train_loader:
        batch_input = batch_input.to(device)
        batch_target = batch_target.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(batch_input)  # (B, T, vocab_size)
        
        # Flatten for CrossEntropyLoss
        loss = criterion(
            logits.reshape(-1, vocab_size),
            batch_target.reshape(-1)
        )
        
        loss.backward()
        
        # Gradient clipping — critical for transformer stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch_input, batch_target in val_loader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)
            
            logits = model(batch_input)
            loss = criterion(
                logits.reshape(-1, vocab_size),
                batch_target.reshape(-1)
            )
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    perplexity = np.exp(avg_val_loss)
    val_perplexities.append(perplexity)
    
    elapsed = time.time() - start_time
    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | '
          f'LR: {current_lr:.1e} | '
          f'Train Loss: {avg_train_loss:.3f} | '
          f'Val Loss: {avg_val_loss:.3f} | '
          f'Perplexity: {perplexity:.1f} | '
          f'Time: {elapsed:.0f}s')

total_time = time.time() - start_time
print(f'\nTraining complete! Total time: {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'Final validation perplexity: {val_perplexities[-1]:.1f}')

### Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(val_losses, label='Val Loss', linewidth=2, color='coral')
axes[0].set_title('Loss Curves', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity
axes[1].plot(val_perplexities, linewidth=2, color='seagreen')
axes[1].axhline(y=vocab_size, color='gray', linestyle='--', alpha=0.5,
                label=f'Random guessing ({vocab_size})')
axes[1].set_title('Validation Perplexity', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate schedule
lrs = [get_lr(e, warmup_epochs, N_EPOCHS) for e in range(N_EPOCHS)]
axes[2].plot(lrs, linewidth=2, color='purple')
axes[2].axvline(x=warmup_epochs-1, color='red', linestyle='--', alpha=0.5, label='End of warmup')
axes[2].set_title('Learning Rate Schedule', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Initial perplexity: {val_perplexities[0]:.1f}')
print(f'Final perplexity:   {val_perplexities[-1]:.1f}')
print(f'Improvement:        {val_perplexities[0] - val_perplexities[-1]:.1f} points')
print()
if val_perplexities[-1] < 8:
    print('🍾 Excellent! Perplexity < 8 — the transformer learned strong patterns!')
elif val_perplexities[-1] < 15:
    print('👍 Good! Try training for more epochs or increasing model size.')
else:
    print('📚 Model could benefit from more training or larger architecture.')

---
## Part 7: Generating Text

Same interface as Project 09, but now using the transformer's autoregressive generation. The key difference: the transformer processes the **entire generated history** at every step (with causal masking to prevent looking ahead).

This is why long generations get slower and slower — each new token requires reprocessing the full sequence.

In [ ]:
@torch.no_grad()
def generate_text(model, start_text, char_to_idx, idx_to_char,
                  length=500, temperature=0.8, device=device):
    """Generate text autoregressively using the transformer.
    
    Unlike the LSTM, the transformer processes the ENTIRE sequence at each step.
    The causal mask ensures it doesn't cheat by looking at future tokens.
    """
    model.eval()
    
    # Convert start text to IDs
    input_ids = [char_to_idx.get(ch, 0) for ch in start_text]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    generated = list(start_text)
    
    for _ in range(length):
        # Truncate to max_seq_len if needed (transformer has a fixed context window)
        if input_tensor.size(1) > model.max_seq_len:
            input_tensor = input_tensor[:, -model.max_seq_len:]
        
        # Forward pass: get predictions for ALL positions
        logits = model(input_tensor)  # (1, seq_len, vocab_size)
        
        # We only care about the prediction for the LAST position
        logits = logits[:, -1, :]  # (1, vocab_size)
        
        # Apply temperature
        logits = logits / temperature
        
        # Convert to probabilities and sample
        probs = torch.softmax(logits, dim=-1)
        next_char_id = torch.multinomial(probs, num_samples=1).item()
        
        # Append to generated text
        generated.append(idx_to_char[next_char_id])
        
        # Append to input for next iteration (growing sequence!)
        next_token = torch.tensor([[next_char_id]], dtype=torch.long).to(device)
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
    
    return ''.join(generated)

print('Generation function ready!')
print()
print('Warning: Transformer generation is slower than LSTM because')
print('it reprocesses the entire sequence at every step (O(n²) in sequence length).')
print('This is why production LLMs use KV-caching to avoid recomputation.')

### Generate with Different Seeds

In [ ]:
seed_texts = [
    "ROMEO: ",
    "JULIET: ",
    "HAMLET: ",
    "To be, or not to be",
]

print("=" * 70)
print("TRANSFORMER-GENERATED SHAKESPEARE")
print("=" * 70)

for seed in seed_texts:
    print(f"\n{'─' * 70}")
    print(f"SEED: {repr(seed)}")
    print(f"{'─' * 70}")
    generated = generate_text(model, seed, char_to_idx, idx_to_char,
                              length=400, temperature=0.7)
    print(generated)
    print()

### Temperature Comparison

In [ ]:
seed = "ROMEO: "
temperatures = [0.2, 0.5, 0.8, 1.2, 2.0]

print("=" * 70)
print(f"TEMPERATURE COMPARISON — Seed: {repr(seed)}")
print("=" * 70)

for temp in temperatures:
    generated = generate_text(model, seed, char_to_idx, idx_to_char,
                              length=200, temperature=temp)
    unique_ratio = len(set(generated)) / len(generated) * 100
    
    print(f"\n--- Temperature = {temp} ---")
    print(generated[:250])
    print(f"(Unique char ratio: {unique_ratio:.1f}%)")

---
## Part 8: Understanding What the Model Learned

### Visualizing Attention Patterns

We can extract the attention weights from our model and see which characters attend to which other characters. This is one of the great advantages of transformers over LSTMs — **interpretability**.

In [ ]:
def get_attention_maps(model, text, char_to_idx, layer_idx=-1, head_idx=None):
    """Extract attention weights from a specific layer/head.
    This requires modifying the forward pass to return attention weights.
    For now, we'll hook into the model to capture them."""
    
    attention_maps = {}
    
    def hook_fn(module, input, output):
        # We need to capture intermediate values — for visualization purposes
        # This is a simplified version
        pass
    
    # For a full implementation, we'd register hooks on the attention layers
    print('In a full implementation, register forward hooks on MultiHeadAttention')
    print('to capture the softmax attention weights, then visualize them as heatmaps.')

print('Attention visualization scaffold ready.')
print()
print('Expected patterns in a trained model:')
print('  - Early layers: attend to nearby characters (local patterns)')
print('  - Middle layers: attend to syntactic roles (subject, verb, object)')
print('  - Later layers: attend to semantic relationships')
print('  - Some heads specialize in previous occurrences of the same word')

---
## Part 9: LSTM vs Transformer — Direct Comparison

Since we're using the same dataset and same task as Project 09, we can compare directly.

| Aspect | LSTM (Project 09) | Transformer (This Project) |
|--------|-------------------|---------------------------|
| **Architecture** | Recurrent — sequential processing | Attention-based — parallel processing |
| **How it sees context** | Compressed hidden state (bottleneck) | Direct attention to every past token |
| **Long-range dependencies** | Struggles beyond ~50 steps | Strong at any distance in context |
| **Training speed** | Slow (sequential, can't parallelize) | Fast (processes all positions at once) |
| **Generation speed** | Fast per step (O(1) state update) | Slow per step (O(n²) — reprocesses whole sequence) |
| **Interpretability** | Hidden state is opaque | Attention weights show what the model "sees" |
| **Training stability** | Can suffer from vanishing/exploding gradients | More stable with layer norm + residuals |
| **Scaling** | Hits diminishing returns | Scales remarkably well (the "scaling laws") |
| **Parameters (our models)** | ~1M | ~3-5M |

### Why Transformers Won

1. **Parallelization** — train on entire sequences at once, not token by token
2. **No gradient bottlenecks** — attention is a direct connection, not a compressed state
3. **Scaling** — bigger models reliably get better (LSTMs hit a wall)
4. **Multi-head attention** — multiple relationship types learned simultaneously

---
## What You Built

1. **Self-Attention from scratch** — Q, K, V projections, scaled dot-product, causal masking
2. **Multi-Head Attention** — multiple attention patterns running in parallel
3. **Positional Encoding** — sinusoidal encoding so the model knows token order
4. **Transformer Blocks** — attention + feed-forward with residual connections and layer norm
5. **A complete GPT-style decoder-only model** — same architecture as GPT-2 (just smaller)
6. **Training with warmup + cosine decay** — the standard transformer training recipe
7. **Autoregressive text generation** — same interface as Project 09

## Key Concepts

| Concept | Meaning |
|---------|--------|
| **Self-Attention** | Every token computes relevance scores with every other token |
| **Q, K, V** | Query (what am I looking for?), Key (what do I contain?), Value (what do I pass?) |
| **Scaled Dot-Product** | Attention score = Q·K^T / √d_k (preventing extreme softmax values) |
| **Causal Mask** | Upper-triangular matrix of -inf that prevents cheating by looking ahead |
| **Multi-Head** | Run N attention patterns in parallel, concatenate, project |
| **Positional Encoding** | Sine/cosine waves injected into embeddings so model knows position |
| **Residual Connection** | x = x + f(x) — lets gradients flow directly through deep networks |
| **Layer Norm** | Normalizes across features (unlike Batch Norm which normalizes across batch) |
| **Decoder-Only** | Only has self-attention (no encoder-decoder cross-attention) — this is what GPT uses |
| **Weight Tying** | Sharing weights between embedding and output layers — saves parameters |

## How This Connects to Real GPT Models

| Our MiniGPT | GPT-2 | GPT-3 | GPT-4 |
|-------------|-------|-------|-------|
| Character tokens | BPE subword tokens | BPE subword tokens | BPE subword tokens |
| ~65 token vocab | ~50K token vocab | ~50K token vocab | ~100K token vocab |
| 6 layers | 12-48 layers | 96 layers | ~120 layers (estimated) |
| 8 heads | 12-25 heads | 96 heads | Unknown |
| 256 embed dim | 768-1600 | 12288 | Unknown |
| ~3-5M params | 124M-1.5B | 175B | ~1.7T (estimated) |
| **Same architecture** | **Same architecture** | **Same architecture** | **Same architecture** |

The core building blocks you just implemented — self-attention, multi-head attention, layer norm, residual connections, causal masking — are **exactly what powers every modern LLM**.

The difference between MiniGPT and GPT-4 is primarily **scale**: more layers, wider embeddings, more heads, trained on more data for longer. The underlying architecture is the same family.

---
## Things to Try

1. **Compare with your LSTM** — train both for the same number of epochs. Which gets lower perplexity?
2. **Increase model size** — try 8 layers, 12 heads. How does perplexity improve?
3. **Longer sequences** — increase `SEQUENCE_LENGTH` to 256 or 512 (warning: memory usage is O(n²))
4. **Train on different text** — your own writing, Wikipedia articles, code
5. **Add dropout to attention weights** — does it help prevent overfitting?
6. **Implement KV-caching** — store computed K and V from previous steps to speed up generation
7. **Try learned positional embeddings** — compare sinusoidal vs learned position embeddings
8. **Add more training data** — combine Shakespeare with other Project Gutenberg texts
9. **Word-level transformer** — use the BPE tokenizer from Project 08 with the transformer architecture
10. **Visualize attention** — hook into the attention layers and plot which characters attend to which

---
## Bonus: Save and Reload Your Model

In [ ]:
model_dir = Path.home() / "LocalAI" / "models" / "mini_gpt"
model_dir.mkdir(parents=True, exist_ok=True)

checkpoint = {
    'model_state_dict': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size,
    'chars': chars,
    'model_config': {
        'embed_dim': 256,
        'n_heads': 8,
        'n_layers': 6,
        'expansion_factor': 4,
        'max_seq_len': SEQUENCE_LENGTH,
        'dropout': 0.1,
    },
    'val_perplexity': val_perplexities[-1] if val_perplexities else None,
}

torch.save(checkpoint, model_dir / 'shakespeare_minigpt.pt')
print(f'Model saved to {model_dir / "shakespeare_minigpt.pt"}')
print()
print('To reload:')
print("""
checkpoint = torch.load(model_dir / 'shakespeare_minigpt.pt', map_location=device)
model = MiniGPT(vocab_size=checkpoint['vocab_size'], **checkpoint['model_config']).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
""")

# Side-by-side comparison with Project 09 LSTM if you saved it
lstm_path = Path.home() / "LocalAI" / "models" / "text_generation" / "shakespeare_lstm.pt"
if lstm_path.exists():
    print()
    print('=' * 60)
    print('LSTM model from Project 09 found!')
    print(f'Location: {lstm_path}')
    print('You can load both models and compare their outputs side by side.')
    print('=' * 60)

---
## Bonus 2: Interactive Generation

Write your own seeds and see what the transformer writes!

In [ ]:
custom_seeds = [
    "The king spoke: ",
    "My dearest love, ",
    "Upon the battlefield, ",
    "I do solemnly swear ",
    "All the world's a stage, ",
]

print("=" * 70)
print("YOUR CUSTOM GENERATIONS")
print("=" * 70)

for seed in custom_seeds:
    print(f"\n{'─' * 70}")
    generated = generate_text(model, seed, char_to_idx, idx_to_char,
                              length=500, temperature=0.8)
    print(generated)
    print()

---
## Where to Go From Here

You've now built the two most important architectures for sequence modeling:
- **Project 09:** LSTM (recurrent)
- **Project 10:** Transformer (attention-based)

Natural next steps in your ML journey:

| Project | What You'd Build | Key Learning |
|---------|-----------------|--------------|
| **Fine-tuning** | Take a pre-trained GPT-2 and fine-tune it on your own text | Transfer learning for language models |
| **Advanced RAG** | Full RAG pipeline with chunking, embedding, and a vector database | Production-grade retrieval |
| **Text Classification with BERT** | Use a pre-trained BERT for classification tasks | Encoder-only transformers |
| **Image Generation** | Build a simple diffusion model or VAE | Generative models beyond text |
| **Reinforcement Learning** | Train an agent to play a game | A completely different learning paradigm |
| **Speech / Audio** | Process audio with transformers | Beyond text and images |

## Resources

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — the original transformer paper
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — Jay Alammar's excellent visual guide
- [The Annotated Transformer](http://nlp.seas.harvard.edu/annotated-transformer/) — line-by-line implementation walkthrough
- [nanoGPT](https://github.com/karpathy/nanoGPT) — Andrej Karpathy's minimal GPT implementation (our design is inspired by this)
- [Let's Build GPT from Scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY) — Karpathy's video walkthrough

In [ ]:
# Final summary
print("=" * 70)
print("PROJECT 10 COMPLETE")
print("=" * 70)
print()
print(f'You built: A {total_params:,}-parameter GPT-style transformer')
print(f'Architecture: {6} layers, {8} heads, {256}-dim embeddings')
print(f'Dataset: Shakespeare (~{len(text):,} characters)')
print(f'Task: Next-character prediction (same as Project 09)')
print()
print('Components you implemented from scratch:')
print('  ✓ Multi-head self-attention with causal masking')
print('  ✓ Positional encoding (sinusoidal)')
print('  ✓ Feed-forward network with GELU')
print('  ✓ Transformer block with residual connections + layer norm')
print('  ✓ Weight tying between embedding and output layers')
print('  ✓ Warmup + cosine decay learning rate schedule')
print()
print('This is the same architecture that powers ChatGPT, Claude, and Gemini.')
print('You now understand what every modern LLM is doing under the hood. 🎉')